In [1]:
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torchvision import transforms
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import zipfile

In [2]:
class args:
    dataset_dir = "/kaggle/input/datasets/almiraraisa/aptos-2019-224px-v2"
    output_dir  = "/kaggle/working/aptos-2019-augmented"
    aug_copies  = 5
    seed        = 35
    image_size  = 224
    mixup_alpha = 0.4

In [3]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

In [4]:
SPATIAL_COLOR_AUG = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=360, border_mode=0, p=0.9), 
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.10,
        rotate_limit=0,
        border_mode=0,
        p=0.5,
    ),

    A.HueSaturationValue(
        hue_shift_limit=10,
        sat_shift_limit=20,
        val_shift_limit=20,
        p=0.7,
    ),
    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2,
        p=0.7,
    ),

    A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.4),
    A.OpticalDistortion(distort_limit=0.2, shift_limit=0.05, p=0.4),
])

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_57/3742026253.py:26: UserWarning: Argument(s) 'shift_limit' are not valid for transform OpticalDistortion
  A.OpticalDistortion(distort_limit=0.2, shift_limit=0.05, p=0.4),


In [5]:
NORMALIZE = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)

TRAIN_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    NORMALIZE,
])

In [6]:
def mixup_pair(
    img_a: np.ndarray,
    label_a: int,
    img_b: np.ndarray,
    label_b: int,
    alpha: float,
) -> tuple[np.ndarray, float]:
    lam = np.random.beta(alpha, alpha)
    mixed = (lam * img_a.astype(np.float32) +
             (1.0 - lam) * img_b.astype(np.float32))
    mixed = np.clip(mixed, 0, 255).astype(np.uint8)
    soft_label = lam * label_a + (1.0 - lam) * label_b
    return mixed, soft_label

In [7]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def load_image(path: Path) -> np.ndarray:
    """Load image as HWC uint8 numpy array (RGB)."""
    return np.array(Image.open(path).convert("RGB"))


def save_image(arr: np.ndarray, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(arr).save(path, format="PNG")


def resolve_fname(name: str) -> str:
    return name if name.endswith(".png") else name + ".png"

In [8]:
def process_val_or_test(
    df: pd.DataFrame,
    src_img_dir: Path,
    dst_img_dir: Path,
    split_name: str,
) -> pd.DataFrame:
    rows = []
    img_col = "id_code" if "id_code" in df.columns else df.columns[0]

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Copying {split_name}"):
        fname = resolve_fname(str(row[img_col]))
        src   = src_img_dir / fname
        dst   = dst_img_dir / fname

        if not dst.exists():
            shutil.copy2(src, dst)

        rows.append({img_col: Path(fname).stem, "diagnosis": float(row["diagnosis"])})

    return pd.DataFrame(rows)

In [9]:
def process_train(
    df: pd.DataFrame,
    src_img_dir: Path,
    dst_img_dir: Path,
    aug_copies: int,
    seed: int,
    mixup_alpha: float,
) -> pd.DataFrame:
    img_col = "id_code" if "id_code" in df.columns else df.columns[0]
    print("Pre-loading training images for MixUp…")
    all_imgs: list[np.ndarray] = []
    all_labels: list[int] = []
    all_stems: list[str] = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Loading"):
        fname = resolve_fname(str(row[img_col]))
        all_imgs.append(load_image(src_img_dir / fname))
        all_labels.append(int(row["diagnosis"]))
        all_stems.append(Path(fname).stem)

    n = len(all_imgs)
    rows = []

    for idx in tqdm(range(n), desc="Augmenting train"):
        img       = all_imgs[idx]
        label     = all_labels[idx]
        stem      = all_stems[idx]

        dst = dst_img_dir / f"{stem}_orig.png"
        save_image(img, dst)
        rows.append({img_col: f"{stem}_orig", "diagnosis": float(label)})

        for copy_idx in range(1, aug_copies + 1):
            det_seed = seed ^ (idx * 1000 + copy_idx)
            random.seed(det_seed)
            np.random.seed(det_seed)

            aug_img = SPATIAL_COLOR_AUG(image=img)["image"]

            dst = dst_img_dir / f"{stem}_aug{copy_idx:02d}.png"
            save_image(aug_img, dst)
            rows.append({img_col: f"{stem}_aug{copy_idx:02d}", "diagnosis": float(label)})

        mix_seed = seed ^ (idx * 10000 + 9999)
        np.random.seed(mix_seed)
        partner_idx = (idx + 1 + np.random.randint(0, n - 1)) % n  # never self
        mixed_img, soft_label = mixup_pair(
            img, label,
            all_imgs[partner_idx], all_labels[partner_idx],
            alpha=mixup_alpha,
        )
        dst = dst_img_dir / f"{stem}_mixup.png"
        save_image(mixed_img, dst)
        rows.append({img_col: f"{stem}_mixup", "diagnosis": soft_label})

    return pd.DataFrame(rows)

In [10]:
def main():
    set_seed(args.seed)

    dataset_dir = Path(args.dataset_dir)
    output_dir  = Path(args.output_dir)
    src_img_dir = dataset_dir / "images"
    dst_img_dir = output_dir  / "images"
    dst_img_dir.mkdir(parents=True, exist_ok=True)

    train_df = pd.read_csv(dataset_dir / "train_split.csv")
    val_df   = pd.read_csv(dataset_dir / "val_split.csv")
    test_df  = pd.read_csv(dataset_dir / "test_split.csv")

    total_train = len(train_df) * (args.aug_copies + 1 + 1)

    print(f"\nDataset root : {dataset_dir}")
    print(f"Output root  : {output_dir}")
    print(f"Train rows   : {len(train_df)}  →  {total_train} after augmentation")
    print(f"  ({args.aug_copies} augmented + 1 original + 1 MixUp per image)")
    print(f"Val rows     : {len(val_df)}")
    print(f"Test rows    : {len(test_df)}\n")

    aug_train_df = process_train(
        train_df, src_img_dir, dst_img_dir,
        args.aug_copies, args.seed, args.mixup_alpha,
    )
    aug_val_df  = process_val_or_test(val_df,  src_img_dir, dst_img_dir, "val")
    aug_test_df = process_val_or_test(test_df, src_img_dir, dst_img_dir, "test")

    aug_train_df.to_csv(output_dir / "train_split.csv", index=False)
    aug_val_df.to_csv(  output_dir / "val_split.csv",   index=False)
    aug_test_df.to_csv( output_dir / "test_split.csv",  index=False)

    print(f"\n  Train images : {len(aug_train_df)}")
    print(f"  Val images   : {len(aug_val_df)}")
    print(f"  Test images  : {len(aug_test_df)}")
    print(f"  CSVs written to {output_dir}")

    dist = aug_train_df["diagnosis"].round().astype(int).value_counts().sort_index()
    print(f"\n  Class distribution (train, after aug, soft labels rounded):")
    print(dist.to_string())
    print(f"\n  DATASET_DIR = '{output_dir}'")

    zip_path = output_dir.parent / "aptos_augmented.zip"
    print(f"\n  Zipping output to {zip_path} …")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for file in output_dir.rglob("*"):
            if file.is_file():
                zf.write(file, file.relative_to(output_dir))
    print(f"  Process complete.")

In [11]:
if __name__ == "__main__":
    main()


Dataset root : /kaggle/input/datasets/almiraraisa/aptos-2019-224px-v2
Output root  : /kaggle/working/aptos-2019-augmented
Train rows   : 2929  →  20503 after augmentation
  (5 augmented + 1 original + 1 MixUp per image)
Val rows     : 366
Test rows    : 367

Pre-loading training images for MixUp…


Copying test: 100%|██████████| 367/367 [00:02<00:00, 144.79it/s]



  Train images : 20503
  Val images   : 366
  Test images  : 367
  CSVs written to /kaggle/working/aptos-2019-augmented

  Class distribution (train, after aug, soft labels rounded):
diagnosis
0    9847
1    2374
2    5575
3    1127
4    1580

  DATASET_DIR = '/kaggle/working/aptos-2019-augmented'

  Zipping output to /kaggle/working/aptos_augmented.zip …
  Process complete.
